# Stage 2 Validation Notebook
## Testing retrieval, reflection, and generation quality — one agent at a time

**Test sequence (run in order — each section builds on the last):**
1. Memory seeding — are the right things in the stream at all?
2. Retrieval — do the right memories surface for a given query?
3. Reflection — do reflections accurately synthesise the memories?
4. Generation — does the final response match the real person?
5. Ablations — what happens when you remove each component?

**To change which LLM is used anywhere:** edit the `Config` block in Step 4 below.

---
### Step 1: Install libraries

In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic \
  sentence-transformers \
  "numpy==1.26.4" \
  "pandas==2.2.2" \
  scipy \
  scikit-learn \
  pyyaml \
  gspread

---
### Step 2: Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Spring 2026/INFO 290: GenAI/GenAI Project 290/berkeley-homes-wildfire-agent-simulation"

In [ ]:
import os
os.chdir(PROJECT_PATH)

---
### Step 3: Imports

In [ ]:
import sys
import json
import yaml
from pathlib import Path
from datetime import datetime

sys.path.insert(0, PROJECT_PATH)

from src.llm.client import Config, init_clients, embed, decide, reflect, score_importance, judge_retrieval, judge_generation
from src.agents.memory import MemoryStream
from src.agents.retrieval import RetrievalConfig, retrieve_memories, pretty_print_retrieval
from src.agents.reflection import ReflectionConfig, maybe_reflect
from src.agents.prompts import build_system_prompt, build_decision_prompt, frame_event, DECISION_QUESTION, pretty_print_prompt

---
### Step 4: LLM Config

**This is the single place to change which models are used for each role.**

| Role | Controls | Options |
|---|---|---|
| `DECISION_MODEL` | Agent response generation | `claude-sonnet-4-6`, `gpt-4o`, `deepseek-r1` |
| `REFLECTION_MODEL` | Belief synthesis | `claude-opus-4-6` (quality), `claude-sonnet-4-6` (cheaper) |
| `JUDGE_MODEL` | Evaluation scoring | `claude-opus-4-6` (recommended — strong reasoning) |

In [ ]:
config = Config(
    DECISION_MODEL    = "claude-sonnet-4-6",
    REFLECTION_MODEL  = "claude-sonnet-4-6",   # swap to claude-opus-4-6 for higher quality
    JUDGE_MODEL       = "claude-opus-4-6",      # keep Opus for evaluation
)

print("LLM config:")
print(f"  Decision:   {config.DECISION_MODEL}")
print(f"  Reflection: {config.REFLECTION_MODEL}")
print(f"  Judge:      {config.JUDGE_MODEL}")

---
### Step 5: Connect to LLM APIs

In [ ]:
client_anthropic = init_clients()

---
### Step 6: Load Margaret


In [ ]:
AGENT_CONFIG_PATH = Path(PROJECT_PATH) / 'config' / 'agents' / 'margaret.yaml'
with open(AGENT_CONFIG_PATH, 'r') as f:
    margaret_config = yaml.safe_load(f)

print(f"Loaded agent: {margaret_config['name']}")
print(f"Held-out scenarios available: {list(margaret_config['held_out_responses'].keys())}")


---
## Section 1: Memory Seeding Evaluation

**Question:** Are the right things in Margaret's memory stream before any events happen?

The judge reviews the full seeded memory stream against each held-out scenario
and assesses whether the seeds give Margaret enough context to respond in character.

In [ ]:
# Initialise and seed Margaret's memory stream
margaret_memory = MemoryStream(margaret_config['name'])
margaret_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)
margaret_memory.pretty_print()


In [ ]:
# Judge: for each held-out scenario, does the seeded memory stream give Margaret
# enough context to respond authentically?

def judge_memory_seeding(client_anthropic, config, memory_stream, scenario_name, scenario):
    """Ask the judge whether the memory stream adequately prepares the agent for this scenario."""
    import json
    all_text = "\n".join(
        f"[{i+1}] Day {m.timestamp} | {m.type} | importance {m.importance}/10 | {m.description}"
        for i, m in enumerate(memory_stream.get_all())
    )
    system = "You are an expert evaluator assessing whether a generative agent's seeded memories are adequate."
    user = (
        f"AGENT CONTEXT: {scenario['context']}\n\n"
        f"SCENARIO: {scenario_name}\n"
        f"REAL RESPONSE: {scenario['real_response']}\n\n"
        f"SEEDED MEMORY STREAM:\n{all_text}\n\n"
        "Assess whether these seed memories give the agent enough context to respond authentically.\n"
        "Respond in this exact JSON format:\n"
        "{\n"
        '  "adequacy_score": <integer 1-10>,\n'
        '  "missing_context": [<what important context is missing>, ...],\n'
        '  "assessment": "<1-2 sentence summary>"\n'
        "}"
    )
    message = client_anthropic.messages.create(
        model=config.JUDGE_MODEL,
        max_tokens=config.JUDGE_MAX_TOKENS,
        temperature=config.JUDGE_TEMPERATURE,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {"adequacy_score": None, "missing_context": [], "assessment": raw}


print("=" * 60)
print("MEMORY SEEDING EVALUATION")
print("=" * 60)
for scenario_name, scenario in margaret_config['held_out_responses'].items():
    result = judge_memory_seeding(client_anthropic, config, margaret_memory, scenario_name, scenario)
    print(f"\nScenario: {scenario_name}")
    print(f"  Adequacy score: {result.get('adequacy_score')}/10")
    print(f"  Assessment: {result.get('assessment')}")
    if result.get('missing_context'):
        print(f"  Missing context:")
        for item in result['missing_context']:
            print(f"    - {item}")

---
## Section 2: Retrieval Evaluation

**Question:** For a given intervention, do the right memories surface?

We test multiple `RetrievalConfig` variants against each scenario.
The judge scores each retrieval set and identifies missed memories.

**To add a new variant:** copy a config line and change the parameters.

In [ ]:
# ── Define retrieval variants to compare ──────────────────────────────────────
# Edit these freely — add / remove rows to test different settings

retrieval_variants = [
    RetrievalConfig(top_k=5,  recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode="dense"),
    RetrievalConfig(top_k=8,  recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode="dense"),
    RetrievalConfig(top_k=8,  recency_weight=0.5, importance_weight=1.0, relevance_weight=2.0, retrieval_mode="dense"),
    RetrievalConfig(top_k=8,  recency_weight=1.0, importance_weight=2.0, relevance_weight=1.0, retrieval_mode="dense"),
    RetrievalConfig(top_k=8,  recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode="sparse"),
    RetrievalConfig(top_k=8,  recency_weight=1.0, importance_weight=1.0, relevance_weight=1.0, retrieval_mode="hybrid", sparse_weight=0.3),
]

print(f"Testing {len(retrieval_variants)} retrieval configurations")

In [ ]:
# ── Define the intervention scenarios to test retrieval against ───────────────
# These are the queries we're retrieving for — aligned with held-out eval scenarios

retrieval_scenarios = {
    scenario_name: scenario['context'] + " " + scenario['real_response']
    for scenario_name, scenario in margaret_config['held_out_responses'].items()
}

print("Retrieval scenarios:")
for name, query in retrieval_scenarios.items():
    print(f"  {name}: '{query[:80]}...'")

In [ ]:
# ── Run retrieval eval: all variants × all scenarios ─────────────────────────

retrieval_results = []  # log for comparison

for scenario_name, query_text in retrieval_scenarios.items():
    query_embed = embed(query_text)
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")

    for rc in retrieval_variants:
        retrieved = retrieve_memories(margaret_memory, query_embed, query_text, rc, current_day=0)
        scores = judge_retrieval(
            client_anthropic, config,
            all_memories=margaret_memory.get_all(),
            intervention=query_text,
            retrieved_memories=retrieved,
        )
        result = {
            "scenario": scenario_name,
            "config": rc.label(),
            "relevance_score": scores.get("relevance_score"),
            "missed_memories": scores.get("missed_memories", []),
            "critique": scores.get("critique"),
        }
        retrieval_results.append(result)
        print(f"\n  Config: {rc.label()}")
        print(f"  Relevance score: {result['relevance_score']}/10")
        print(f"  Critique: {result['critique']}")
        if result['missed_memories']:
            print(f"  Missed: {result['missed_memories']}")

print("\nRetrieval evaluation complete.")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

print(f"\n{'Scenario':<30} {'Config':<60} {'Score':>6}")
print("-" * 100)
for r in sorted(retrieval_results, key=lambda x: (x['scenario'], -(x['relevance_score'] or 0))):
    print(f"{r['scenario']:<30} {r['config']:<60} {str(r['relevance_score']):>6}/10")

# Pick the best config for each scenario (highest relevance score)
by_scenario = {}
for r in retrieval_results:
    s = r['scenario']
    if s not in by_scenario or (r['relevance_score'] or 0) > (by_scenario[s]['relevance_score'] or 0):
        by_scenario[s] = r

print("\nBest config per scenario:")
for scenario, best in by_scenario.items():
    print(f"  {scenario}: {best['config']} → {best['relevance_score']}/10")

---
## Section 3: Reflection Evaluation

**Question:** After experiencing some events, do Margaret's reflections accurately
synthesise her memories into coherent higher-level beliefs?

We feed Margaret the baseline scenario events first, then trigger reflection
and evaluate the quality of the resulting beliefs.

In [ ]:
# ── Feed Margaret the baseline events to build up her memory stream ──────────

SCENARIO_PATH = Path(PROJECT_PATH) / 'config' / 'agents' / 'config' / 'scenarios' / 'baseline.yaml'
with open(SCENARIO_PATH, 'r') as f:
    scenario = yaml.safe_load(f)

# Re-seed a fresh memory stream (so this section is independent)
reflection_memory = MemoryStream(margaret_config['name'])
reflection_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

# Use best retrieval config from Section 2 (or set manually here)
best_retrieval_cfg = RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=2.0)

# Process each event: perceive → retrieve → decide → store observation
for event in scenario['events']:
    if margaret_config['name'] not in str(event.get('target_agents', '')) and event.get('target_agents') != 'all':
        continue

    situation = frame_event(event['channel'], event['content'])
    query_embed = embed(situation)
    retrieved = retrieve_memories(reflection_memory, query_embed, situation, best_retrieval_cfg, current_day=event['day'])

    system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
    user_prompt = build_decision_prompt(situation, DECISION_QUESTION)
    decision = decide(client_anthropic, system_prompt, user_prompt)

    # Store the event and decision
    imp = score_importance(client_anthropic, event['content'], margaret_config['seed_narrative'])
    reflection_memory.add(event['content'], imp, embed(event['content']), 'observation', timestamp=event['day'])
    imp_d = score_importance(client_anthropic, decision, margaret_config['seed_narrative'])
    reflection_memory.add(decision[:200], imp_d, embed(decision[:200]), 'decision', timestamp=event['day'])

    print(f"Day {event['day']}: {event['type']} → stored observation + decision")

print(f"\nMemory stream now has {reflection_memory.count()} memories")

In [ ]:
# ── Reflection config variants to test ───────────────────────────────────────
# Edit freely — try different thresholds, question counts, models

reflection_variants = [
    ReflectionConfig(threshold=50.0,  num_questions=3, reflection_importance=8),
    ReflectionConfig(threshold=50.0,  num_questions=5, reflection_importance=8),
]

reflection_results = []

for rc in reflection_variants:
    print(f"\n{'='*60}")
    print(f"Reflection config: threshold={rc.threshold}, num_questions={rc.num_questions}")
    print(f"{'='*60}")

    # Use a copy of the stream so variants don't pollute each other
    import copy
    stream_copy = copy.deepcopy(reflection_memory)

    new_reflections, _ = maybe_reflect(
        stream=stream_copy,
        client_anthropic=client_anthropic,
        config=config,
        reflection_config=rc,
        agent_seed=margaret_config['seed_narrative'],
        current_day=scenario['events'][-1]['day'],
        last_reflection_index=0,
    )

    # Judge: are these reflections accurate and insightful?
    if new_reflections:
        reflections_text = "\n".join(f"[{i+1}] {m.description}" for i, m in enumerate(new_reflections))
        memories_text = "\n".join(
            f"[{i+1}] {m.description}"
            for i, m in enumerate(stream_copy.get_by_type('observation') + stream_copy.get_by_type('decision'))
        )
        system = "You are evaluating whether an agent's reflections accurately represent higher-level beliefs."
        user = (
            f"AGENT CONTEXT: {margaret_config['seed_narrative'][:200]}\n\n"
            f"SOURCE MEMORIES:\n{memories_text}\n\n"
            f"REFLECTIONS GENERATED:\n{reflections_text}\n\n"
            "Score these reflections on insight quality and accuracy.\n"
            "Respond in JSON: {\"insight_score\": <1-10>, \"accuracy_score\": <1-10>, \"critique\": \"<1-2 sentences>\"}"
        )
        import json
        msg = client_anthropic.messages.create(
            model=config.JUDGE_MODEL, max_tokens=512, temperature=0.0,
            system=system, messages=[{"role": "user", "content": user}]
        )
        raw = msg.content[0].text.strip()
        if raw.startswith("```"): raw = raw.split("```")[1]; raw = raw[4:] if raw.startswith("json") else raw
        try:
            scores = json.loads(raw.strip())
        except:
            scores = {"insight_score": None, "accuracy_score": None, "critique": raw}

        reflection_results.append({"config": f"threshold={rc.threshold} questions={rc.num_questions}", **scores})
        print(f"  Insight score:  {scores.get('insight_score')}/10")
        print(f"  Accuracy score: {scores.get('accuracy_score')}/10")
        print(f"  Critique: {scores.get('critique')}")
    else:
        print("  Threshold not met — no reflections triggered.")

---
## Section 4: Generation Evaluation

**Question:** Does Margaret's full simulated response match what the real person said?

We run the complete pipeline (seed → retrieve → decide) for each held-out scenario
and compare the simulated response to the real interview response using the judge.

Use the scores here to tune the decision model and prompt structure.

In [ ]:
# ── Full pipeline: seed → retrieve → decide → judge ──────────────────────────
# Uses the best retrieval config identified in Section 2
# Change best_retrieval_cfg below to test different settings

generation_results = []

# Fresh memory stream for generation eval
gen_memory = MemoryStream(margaret_config['name'])
gen_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

for scenario_name, scenario in margaret_config['held_out_responses'].items():
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")

    query_text = scenario['context']
    query_embed = embed(query_text)
    retrieved = retrieve_memories(gen_memory, query_embed, query_text, best_retrieval_cfg, current_day=0)

    system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
    user_prompt = build_decision_prompt(scenario['context'], DECISION_QUESTION)

    simulated = decide(client_anthropic, system_prompt, user_prompt)

    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=scenario['real_response'],
        context=scenario['context'],
    )

    generation_results.append({"scenario": scenario_name, **scores})

    print(f"Real response:      {scenario['real_response']}")
    print(f"Simulated response: {simulated[:200]}...")
    print(f"\nScores:")
    print(f"  Character fidelity:  {scores.get('character_fidelity')}/10")
    print(f"  Decision alignment:  {scores.get('decision_alignment')}/10")
    print(f"  Reasoning alignment: {scores.get('reasoning_alignment')}/10")
    print(f"  Overall:             {scores.get('overall_score')}/10")
    print(f"  Critique: {scores.get('critique')}")

In [ ]:
# ── Generation summary ────────────────────────────────────────────────────────

print(f"\n{'Scenario':<30} {'Fidelity':>9} {'Decision':>9} {'Reasoning':>10} {'Overall':>8}")
print("-" * 70)
for r in generation_results:
    print(f"{r['scenario']:<30} {str(r.get('character_fidelity')):>8}/10 {str(r.get('decision_alignment')):>8}/10 {str(r.get('reasoning_alignment')):>9}/10 {str(r.get('overall_score')):>7}/10")

---
## Section 4b: Interview-Question Test

**Question:** Given a verbatim interview question, does the agent respond as the real person did?

This differs from Section 4 in two ways:
- The situation is a **direct interview question** (verbatim from the transcript)
- The response prompt asks for **natural conversation**, not a list of actions

Each cell below tests a different scenario. Edit `test_question` to try your own questions.  
Set `matched_scenario` to a key from `margaret_config['held_out_responses']` to score against the real response, or `None` to skip.

**Source:** `data/interview_transcripts/Jennifer - Feb 25 at 1_03 PM.docx`

In [ ]:
# ── Scenario: insurance non-renewal ─────────────────────────────────────────────────────────────
INTERVIEW_QUESTION = (
    "You are being interviewed about your wildfire preparedness experience. "
    "The interviewer says the above. Respond naturally and in character — "
    "speak conversationally as you would in a real interview. "
    "Don't list actions; just talk."
)

test_question    = "We're hearing from the insurance side that some folks managed to keep their insurance and some didn't — they ended up going on the Fair Plan and paying triple what they were before. Has that been a concern for you?"
matched_scenario = "insurance_non_renewal"

interview_memory = MemoryStream(margaret_config['name'])
interview_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

query_embed   = embed(test_question)
retrieved     = retrieve_memories(interview_memory, query_embed, test_question, best_retrieval_cfg, current_day=0)
system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
user_prompt   = build_decision_prompt(test_question, INTERVIEW_QUESTION)
simulated     = decide(client_anthropic, system_prompt, user_prompt)

print(f"Question:\n{test_question}\n")
print(f"{margaret_config['name']}'s response:\n{simulated}\n")

if matched_scenario and matched_scenario in margaret_config['held_out_responses']:
    held_out = margaret_config['held_out_responses'][matched_scenario]
    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=held_out['real_response'],
        context=held_out['context'],
    )
    print(f"Real response:\n{held_out['real_response']}\n")
    print(f"  Source: {held_out.get('source', 'n/a')}\n")
    print('Scores:')
    print(f"  Character fidelity:  {scores.get('character_fidelity')}/10")
    print(f"  Decision alignment:  {scores.get('decision_alignment')}/10")
    print(f"  Reasoning alignment: {scores.get('reasoning_alignment')}/10")
    print(f"  Overall:             {scores.get('overall_score')}/10")
    print(f"  Critique: {scores.get('critique')}")


In [ ]:
# ── Scenario: contractor access / official guidance ─────────────────────────────────────────────────────────────
INTERVIEW_QUESTION = (
    "You are being interviewed about your wildfire preparedness experience. "
    "The interviewer says the above. Respond naturally and in character — "
    "speak conversationally as you would in a real interview. "
    "Don't list actions; just talk."
)

test_question    = "Getting access to contractors who are knowledgeable around this — is that something that you've faced as well?"
matched_scenario = "official_regulatory_notice"

interview_memory = MemoryStream(margaret_config['name'])
interview_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

query_embed   = embed(test_question)
retrieved     = retrieve_memories(interview_memory, query_embed, test_question, best_retrieval_cfg, current_day=0)
system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
user_prompt   = build_decision_prompt(test_question, INTERVIEW_QUESTION)
simulated     = decide(client_anthropic, system_prompt, user_prompt)

print(f"Question:\n{test_question}\n")
print(f"{margaret_config['name']}'s response:\n{simulated}\n")

if matched_scenario and matched_scenario in margaret_config['held_out_responses']:
    held_out = margaret_config['held_out_responses'][matched_scenario]
    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=held_out['real_response'],
        context=held_out['context'],
    )
    print(f"Real response:\n{held_out['real_response']}\n")
    print(f"  Source: {held_out.get('source', 'n/a')}\n")
    print('Scores:')
    print(f"  Character fidelity:  {scores.get('character_fidelity')}/10")
    print(f"  Decision alignment:  {scores.get('decision_alignment')}/10")
    print(f"  Reasoning alignment: {scores.get('reasoning_alignment')}/10")
    print(f"  Overall:             {scores.get('overall_score')}/10")
    print(f"  Critique: {scores.get('critique')}")


In [ ]:
# ── Scenario: DIY zone-zero approach (no held-out — set matched_scenario=None to skip scoring) ─────────────────────────────────────────────────────────────
INTERVIEW_QUESTION = (
    "You are being interviewed about your wildfire preparedness experience. "
    "The interviewer says the above. Respond naturally and in character — "
    "speak conversationally as you would in a real interview. "
    "Don't list actions; just talk."
)

test_question    = "You mentioned you've been working on the home projects yourself. How do you usually do it — do you go and Google stuff and get the materials?"
matched_scenario = None

interview_memory = MemoryStream(margaret_config['name'])
interview_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

query_embed   = embed(test_question)
retrieved     = retrieve_memories(interview_memory, query_embed, test_question, best_retrieval_cfg, current_day=0)
system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
user_prompt   = build_decision_prompt(test_question, INTERVIEW_QUESTION)
simulated     = decide(client_anthropic, system_prompt, user_prompt)

print(f"Question:\n{test_question}\n")
print(f"{margaret_config['name']}'s response:\n{simulated}\n")

if matched_scenario and matched_scenario in margaret_config['held_out_responses']:
    held_out = margaret_config['held_out_responses'][matched_scenario]
    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=held_out['real_response'],
        context=held_out['context'],
    )
    print(f"Real response:\n{held_out['real_response']}\n")
    print(f"  Source: {held_out.get('source', 'n/a')}\n")
    print('Scores:')
    print(f"  Character fidelity:  {scores.get('character_fidelity')}/10")
    print(f"  Decision alignment:  {scores.get('decision_alignment')}/10")
    print(f"  Reasoning alignment: {scores.get('reasoning_alignment')}/10")
    print(f"  Overall:             {scores.get('overall_score')}/10")
    print(f"  Critique: {scores.get('critique')}")


In [ ]:
# ── Scenario: fencing cost and options (no held-out) ─────────────────────────────────────────────────────────────
INTERVIEW_QUESTION = (
    "You are being interviewed about your wildfire preparedness experience. "
    "The interviewer says the above. Respond naturally and in character — "
    "speak conversationally as you would in a real interview. "
    "Don't list actions; just talk."
)

test_question    = "You mentioned needing to do some fencing. What's the situation there — have you had a chance to look into what that would cost or what options exist?"
matched_scenario = None

interview_memory = MemoryStream(margaret_config['name'])
interview_memory.load_seeds(
    seeds=margaret_config['memory_seeds'],
    client_anthropic=client_anthropic,
    agent_seed_narrative=margaret_config['seed_narrative'],
)

query_embed   = embed(test_question)
retrieved     = retrieve_memories(interview_memory, query_embed, test_question, best_retrieval_cfg, current_day=0)
system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
user_prompt   = build_decision_prompt(test_question, INTERVIEW_QUESTION)
simulated     = decide(client_anthropic, system_prompt, user_prompt)

print(f"Question:\n{test_question}\n")
print(f"{margaret_config['name']}'s response:\n{simulated}\n")

if matched_scenario and matched_scenario in margaret_config['held_out_responses']:
    held_out = margaret_config['held_out_responses'][matched_scenario]
    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=held_out['real_response'],
        context=held_out['context'],
    )
    print(f"Real response:\n{held_out['real_response']}\n")
    print(f"  Source: {held_out.get('source', 'n/a')}\n")
    print('Scores:')
    print(f"  Character fidelity:  {scores.get('character_fidelity')}/10")
    print(f"  Decision alignment:  {scores.get('decision_alignment')}/10")
    print(f"  Reasoning alignment: {scores.get('reasoning_alignment')}/10")
    print(f"  Overall:             {scores.get('overall_score')}/10")
    print(f"  Critique: {scores.get('critique')}")


---
## Section 5: Ablations

**Question:** What is each component actually contributing?

We run four conditions and compare generation scores:

| Condition | Memory | Retrieval scoring | Reflection |
|---|---|---|---|
| `full` | ✓ | ✓ (weighted) | ✓ |
| `no_memory` | ✗ | — | — |
| `no_retrieval_scoring` | ✓ | ✗ (recency only) | ✗ |
| `no_reflection` | ✓ | ✓ (weighted) | ✗ |

In [ ]:
def run_ablation(
    scenario_name: str,
    scenario: dict,
    memory_stream: MemoryStream,
    use_memory: bool = True,
    use_retrieval_scoring: bool = True,
    use_reflection: bool = True,
) -> dict:
    """
    Run the full pipeline with each component optionally disabled.

    use_memory=False          → empty memory list in prompt
    use_retrieval_scoring=False → top-k by recency only (all weights = 0 except recency)
    use_reflection=False      → skip reflection even if threshold met
    """
    query_text = scenario['context']
    query_embed = embed(query_text)

    if not use_memory:
        retrieved = []
    elif not use_retrieval_scoring:
        # Recency-only: set importance and relevance weights to 0
        recency_only_cfg = RetrievalConfig(
            top_k=best_retrieval_cfg.top_k,
            recency_weight=1.0,
            importance_weight=0.0,
            relevance_weight=0.0,
        )
        retrieved = retrieve_memories(memory_stream, query_embed, query_text, recency_only_cfg)
    else:
        retrieved = retrieve_memories(memory_stream, query_embed, query_text, best_retrieval_cfg)

    # For this ablation we skip reflection even if it's in the stream
    # (use_reflection=False just means we don't add reflections before running)

    system_prompt = build_system_prompt(margaret_config['seed_narrative'], retrieved)
    user_prompt = build_decision_prompt(query_text, DECISION_QUESTION)
    simulated = decide(client_anthropic, system_prompt, user_prompt)

    scores = judge_generation(
        client_anthropic, config,
        simulated_response=simulated,
        real_response=scenario['real_response'],
        context=scenario['context'],
    )
    return {"scenario": scenario_name, **scores, "simulated": simulated}

In [ ]:
# ── Run all four ablation conditions ─────────────────────────────────────────
# Note: for no_reflection we use reflection_memory (has no reflections in stream)
# For full we would add reflections — here we keep it simple with just the baseline stream

ablation_conditions = [
    {"label": "full",                 "use_memory": True,  "use_retrieval_scoring": True,  "use_reflection": True},
    {"label": "no_memory",            "use_memory": False, "use_retrieval_scoring": False, "use_reflection": False},
    {"label": "no_retrieval_scoring", "use_memory": True,  "use_retrieval_scoring": False, "use_reflection": False},
    {"label": "no_reflection",        "use_memory": True,  "use_retrieval_scoring": True,  "use_reflection": False},
]

ablation_results = []

for scenario_name, scenario in margaret_config['held_out_responses'].items():
    print(f"\nScenario: {scenario_name}")
    for condition in ablation_conditions:
        result = run_ablation(
            scenario_name, scenario, gen_memory,
            use_memory=condition['use_memory'],
            use_retrieval_scoring=condition['use_retrieval_scoring'],
            use_reflection=condition['use_reflection'],
        )
        result['condition'] = condition['label']
        ablation_results.append(result)
        print(f"  {condition['label']:<25} → overall {result.get('overall_score')}/10")

In [ ]:
# ── Ablation summary table ────────────────────────────────────────────────────

print(f"\n{'Condition':<25} {'Scenario':<30} {'Overall':>8} {'Fidelity':>9} {'Decision':>9}")
print("-" * 85)
for r in ablation_results:
    print(
        f"{r['condition']:<25} {r['scenario']:<30} "
        f"{str(r.get('overall_score')):>7}/10 "
        f"{str(r.get('character_fidelity')):>8}/10 "
        f"{str(r.get('decision_alignment')):>8}/10"
    )

---
## Results: JSONL, CSV, and Google Sheets

Results are saved three ways:
1. **JSONL** — one entry per eval run, full fidelity (nested fields preserved)
2. **CSV** — flat table, easy to open in Excel / Numbers
3. **Google Sheet** — shareable with your team; each result type gets its own tab

Each run gets a timestamped `run_id` so you can compare runs across sessions.

In [ ]:
import pandas as pd

output_dir = Path(PROJECT_PATH) / 'outputs' / 'eval'
output_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')

# ── Build flat result rows ────────────────────────────────────────────────────
all_results = (
    [{"type": "retrieval",  "run_id": run_id, **r} for r in retrieval_results]
    + [{"type": "generation", "run_id": run_id, **r} for r in generation_results]
    + [{"type": "ablation",   "run_id": run_id, **r} for r in ablation_results]
)

# Lists → strings for CSV/Sheets compatibility
def flatten_row(r):
    out = {k: v for k, v in r.items() if k not in ('embedding', 'simulated')}
    if isinstance(out.get('missed_memories'), list):
        out['missed_memories'] = '; '.join(out['missed_memories'])
    return out

flat_results = [flatten_row(r) for r in all_results]

# ── Save JSONL (full fidelity) ────────────────────────────────────────────────
jsonl_path = output_dir / f"stage2_eval_{run_id}.jsonl"
with open(jsonl_path, 'w') as f:
    for r in all_results:
        safe = {k: v for k, v in r.items() if k != 'embedding'}
        f.write(json.dumps(safe) + '\n')
print(f"JSONL saved: {jsonl_path}")

# ── Save CSV (flat) ───────────────────────────────────────────────────────────
df = pd.DataFrame(flat_results)
csv_path = output_dir / f"stage2_eval_{run_id}.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved:  {csv_path}")
print(f"Total rows: {len(df)}")

# ── Display inline ────────────────────────────────────────────────────────────
from IPython.display import display

score_cols = ['character_fidelity', 'decision_alignment', 'reasoning_alignment',
              'overall_score', 'relevance_score', 'insight_score', 'accuracy_score', 'adequacy_score']

for result_type in df['type'].unique():
    subset = df[df['type'] == result_type].copy()
    # Only show columns that have data for this type
    cols = ['type', 'scenario'] + [c for c in score_cols if c in subset.columns and subset[c].notna().any()]
    if 'condition' in subset.columns: cols.insert(2, 'condition')
    if 'config' in subset.columns:    cols.insert(2, 'config')
    if 'model' in subset.columns:     cols.insert(2, 'model')
    cols += ['critique']
    display_cols = [c for c in cols if c in subset.columns]
    print(f"\n── {result_type.upper()} ──")
    display(subset[display_cols].reset_index(drop=True))

---
### Optional: Export to Google Sheets

Creates a new Google Sheet with one tab per result type (retrieval / generation / ablation).
The sheet URL is printed — share it with your team or supervisor.

You only need to run this once per session (Colab re-auth is automatic via Drive mount).

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate (uses same Google account as Drive mount — no extra login needed)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Create a new sheet named after this run
sheet_title = f"Stage 2 Validation — {run_id}"
sh = gc.create(sheet_title)

# Write each result type to its own tab
first_tab = True
for result_type, group in df.groupby('type'):
    group_clean = group.fillna('').astype(str)
    headers = group_clean.columns.tolist()
    rows = group_clean.values.tolist()

    if first_tab:
        ws = sh.get_worksheet(0)
        ws.update_title(result_type)
        first_tab = False
    else:
        ws = sh.add_worksheet(title=result_type, rows=len(rows)+5, cols=len(headers)+2)

    ws.update([headers] + rows)
    print(f"  Tab '{result_type}': {len(rows)} rows")

# Make readable by anyone with the link
sh.share('', perm_type='anyone', role='reader')

print(f"\nGoogle Sheet created: {sh.url}")
print("Share this link with your team or supervisor.")